# Segmentos de precio de departamentos en CABA
**Programación Avanzada en Ciencia de Datos — Proyecto integrador**

Clasificación binaria de publicaciones: **segmento alto** (1) frente a **estándar** (0).
La lógica vive en módulos Python para que este documento pueda ejecutarse de principio a fin.
No se realizan operaciones Git ni se publica información.

**Pregunta:** ¿hasta qué punto superficie, ambientes y barrio permiten identificar publicaciones del segmento de precio alto en la muestra de CABA de 2020?

## 1. Parámetros y reproducibilidad
La siguiente celda está etiquetada `parameters` para Papermill. `run_id` separa los resultados de cada ejecución. Repetirlo reemplaza sus archivos; cambiar el cuantil redefine el problema y no constituye una mejora del mismo modelo.

In [ ]:
archivo = "datasets/departamentos_2020.csv"
salida = "resultados"
run_id = "departamentos_2020"
cuantil = 0.75
semilla = 42
folds = 3

In [ ]:
from pathlib import Path
import json
import sqlite3

import pandas as pd
from IPython.display import Image, Markdown, display
from inmuebles.datos import cargar_y_limpiar, separar_y_etiquetar
from inmuebles.flujo import Configuracion, ejecutar

assert Path(archivo).is_file(), "Ejecutar desde la raíz del proyecto"
config = Configuracion(archivo, salida, run_id, cuantil, semilla, folds)
carpeta = Path(salida) / run_id

## 2. Fuente y preparación
Fuente: [BA Data — Departamentos en venta](https://data.buenosaires.gob.ar/dataset/departamentos-venta).
Archivo original de 2020, descargado el 30/10/2026; procedencia y SHA-256 en `datasets/fuentes_y_perfil.json`.
Una fila representa **una publicación**, no una venta efectiva ni una propiedad única.
El archivo no incluye antigüedad. Los precios son históricos y la muestra no es un censo.

Se eliminan duplicados exactos y precios inválidos. Superficies fuera de 10–1000 m² y ambientes fuera de 1–20 se marcan como faltantes, sin reconstruirlos a partir del precio. Son límites de plausibilidad declarados; algunos casos extremos podrían ser legítimos. Se normalizan barrio y dirección. La imputación se aprende después, dentro del Pipeline.

In [ ]:
datos, auditoria = cargar_y_limpiar(archivo)
display(pd.Series(auditoria, name="Cantidad").to_frame())
display(datos[["superficie", "ambientes", "precio_usd"]].describe())

## 3. Separación y definición del objetivo
Se reserva el 20 % de las **direcciones normalizadas** mediante `GroupShuffleSplit`. La proporción de filas puede diferir del 20 %. Las unidades de un mismo edificio quedan juntas; variantes no detectadas de una dirección siguen siendo una limitación.

El umbral se calcula **sólo en entrenamiento**: percentil 75 del precio publicado en USD. La clase positiva exige `precio_usd > umbral`; los empates quedan en estándar. No se interpreta como lujo ni calidad constructiva.

**X:** superficie, ambientes y barrio. **y:** segmento alto. Se excluyen precio en USD, pesos, precio/m², cotización, dirección y etiqueta del conjunto de predictores.

In [ ]:
train, test, umbral = separar_y_etiquetar(datos, cuantil, semilla)
assert set(train.grupo_direccion).isdisjoint(set(test.grupo_direccion))
assert umbral == train.precio_usd.quantile(cuantil)
display(Markdown(f"**Umbral calculado en entrenamiento: USD {umbral:,.0f}.**"))
display(pd.DataFrame({
    "Partición": ["Entrenamiento", "Prueba"],
    "Publicaciones": [len(train), len(test)],
    "Direcciones": [train.grupo_direccion.nunique(), test.grupo_direccion.nunique()],
    "Proporción de segmento alto": [train.segmento_alto.mean(), test.segmento_alto.mean()],
}))

## 4. Entrenamiento y selección
El `ColumnTransformer` imputa numéricas con mediana y aplica `StandardScaler`; las categóricas reciben una etiqueta para faltantes y `OneHotEncoder(handle_unknown="ignore")`.
Todo está dentro del `Pipeline`: cada fold ajusta sus propios parámetros de transformación.

Se usa la misma validación `StratifiedGroupKFold` de tres folds para todos los candidatos, sin compartir direcciones entre ajuste y validación. La estratificación es aproximada por la restricción de grupos.

- **Base:** `DummyClassifier(most_frequent)`.
- **Regresión Logística:** C ∈ {0.1, 1, 10}; ponderación de clases normal o balanceada.
- **Random Forest:** 120 árboles; profundidad 12 o sin límite; mínimo por hoja 5 o 20.

`GridSearchCV` optimiza **F1 de segmento alto**. La elección entre logística y bosque se fija por CV **antes** de evaluar prueba. No se elige usando las métricas del holdout.

In [ ]:
resumen = ejecutar(config)
assert resumen["grupos_compartidos"] == 0
comparacion = pd.read_csv(carpeta / "metricas.csv")
display(comparacion.drop(columns="parametros").round(4))

## 5. Evaluación final
**Accuracy** mide el acierto global. **Precision** indica qué proporción de los predichos altos realmente lo son. **Recall** indica qué proporción de los altos se detecta. **F1** equilibra precision y recall; su prioridad busca evitar que la mayoría estándar domine la evaluación.

Las matrices usan filas = realidad y columnas = predicción, en orden [estándar, alto]. Se evalúan los tres modelos para comparación académica; el ganador continúa siendo el elegido antes por CV. Las medias de CV utilizadas para selección pueden ser optimistas y su desviación no equivale a un intervalo de confianza. El umbral de probabilidad queda en 0.5, sin ajuste sobre prueba.

In [ ]:
ganador = resumen["configuracion"]["modelo_elegido"]
fila = comparacion.set_index("modelo").loc[ganador]
base = comparacion.set_index("modelo").loc["base"]
display(Markdown(
    f"**Elegido por CV: {ganador}.** F1 CV: {fila.f1_cv_mean:.3f} "
    f"(desviación {fila.f1_cv_std:.3f}). F1 de prueba: {fila.f1_test:.3f}. "
    f"Recall de prueba: {fila.recall_test:.3f}. Accuracy base: {base.accuracy_test:.3f}."
))
for nombre in ["03_comparacion.png", "04_matrices_confusion.png"]:
    display(Image(filename=str(carpeta / "graficos" / nombre)))

## 6. Exploración e interpretación
Los gráficos exploratorios usan **entrenamiento**. La muestra de dispersión limita la cantidad de puntos únicamente para legibilidad; todos los datos preparados participan en el modelado.
Las importancias del bosque o magnitudes de coeficientes describen asociaciones predictivas; no prueban causalidad.

In [ ]:
for nombre in ["01_distribuciones.png", "02_exploracion.png", "05_variables.png"]:
    display(Image(filename=str(carpeta / "graficos" / nombre)))

## 7. Integración con SQLite
`datos_limpios` conserva la partición y los valores para auditoría. `datos_preprocesados` contiene las variables numéricas efectivamente utilizadas por el ganador, con transformador ajustado sólo en entrenamiento. `predicciones`, `metricas` y `configuracion` guardan las salidas y sus parámetros. `fila_fuente` vincula las publicaciones con sus predicciones.

In [ ]:
with sqlite3.connect(carpeta / "proyecto.sqlite") as conexion:
    tablas = [r[0] for r in conexion.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    )]
    conteos = {tabla: conexion.execute(
        f'SELECT count(*) FROM "{tabla}"'
    ).fetchone()[0] for tabla in tablas}
    display(pd.Series(conteos, name="Filas").to_frame())
    display(pd.read_sql_query("SELECT * FROM metricas", conexion))
    huerfanas = conexion.execute(
        "SELECT count(*) FROM predicciones p LEFT JOIN datos_limpios d "
        "ON p.fila_fuente=d.fila_fuente WHERE d.fila_fuente IS NULL"
    ).fetchone()[0]
    assert huerfanas == 0
assert conteos["datos_limpios"] == resumen["limpieza"]["filas_limpias"]
assert conteos["datos_preprocesados"] == conteos["datos_limpios"]
assert conteos["predicciones"] == 3 * resumen["prueba"]["filas"]

## 8. Evidencia y límites
La ejecución conserva modelo, CSV, JSON, log, gráficos y base de datos bajo el mismo `run_id`. El modelo se guarda ajustado en entrenamiento, sin volver a entrenarlo con prueba. Las versiones y huella del dataset permiten rastrear esta corrida.

Los resultados describen publicaciones de **2020**, con fuerte concentración en el segundo trimestre. La etiqueta representa un segmento relativo; no significa lujo. Faltan antigüedad, estado, amenities y ubicación precisa. Hay errores de superficie y repetición de avisos. La normalización de direcciones reduce, pero no elimina totalmente, la posibilidad de compartir inmuebles entre grupos.

Una validación para tasaciones actuales necesitaría datos recientes, identificación más confiable de propiedades y una evaluación temporal independiente. No se incorporan nuevas variables ni se cambian hiperparámetros a partir del resultado final de prueba.

In [ ]:
display(pd.Series(resumen["configuracion"]["versiones"], name="Versión").to_frame())
display(Markdown(f"Archivos de esta corrida: `{carpeta}`. "
                 f"Tiempo de ejecución: {resumen['duracion_segundos']:.1f} segundos."))